In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [30]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GOOGLE_API_KEY") is not None)

True


In [38]:
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [39]:
#create a state

class LLMState(TypedDict):
    question : str
    answer: str

In [48]:
def llmqa(state :LLMState) -> LLMState:

    #extract the question from state
    question = state['question']

    #form a prompt
    prompt = f'Answer the following question {question}'

    #ask that question to the LLM
    answer = model.invoke(prompt).content[0]['text']

    #update the answer in the state
    state['answer'] = answer

    return state



In [49]:
# create a graph
graph = StateGraph(LLMState)

#add nodes
graph.add_node('llm_qa', llmqa)

#add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compile
workflow = graph.compile()


In [50]:
# execute

initial_state = {'question': 'How far is moon from the earth?'}


In [51]:
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is moon from the earth?', 'answer': 'The distance from the Earth to the Moon varies because the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle. \n\n* **Average distance:** About **384,400 kilometers** (238,855 miles).\n* **Closest point (Perigee):** About **363,300 kilometers** (225,700 miles).\n* **Farthest point (Apogee):** About **405,500 kilometers** (252,100 miles). \n\nTo put that distance into perspective, you could fit all the planets in our solar system side-by-side in the space between the Earth and the Moon!'}
